# GARCH LSTM statistical and decision focused training

This notebook produces monthly variance forecasts and market or factor exposures for comparing **accuracy training A**
with **decision training B**. The architecture, data, exposure cap and transaction-cost assumptions are held constant
within that comparison. The central question is whether changing the training objective improves the portfolio's
out-of-sample decision objective. Forecasts also support the final analysis of volatility timing against buy and hold.

**No performance conclusion is built into this notebook.** New forecasts must be evaluated with paired inference and
multiple-testing adjustments in the final analysis.

The primary protocol uses the common core French factor history from 1963, a zero conditional mean, five calendar years
of internal validation, fresh model initialization each year, and a forecast ensemble formed before allocation. Parameters
are fitted before validation and selected checkpoints are retained without a subsequent full-history refit.

**How to run:** save this notebook beside the project's `data/` folder. Run all cells in order with `SMOKE_TEST = True`.
That checks one factor, one seed and one forecast year with very few training steps. After it succeeds, set
`SMOKE_TEST = False`, restart the kernel and run all cells for the full experiment. Smoke-run performance is not research evidence.

**Execution status:** Markdown editing does not establish that training ran successfully. The numerical checks in Section 7
must pass in your PyTorch environment before you interpret any results. The supplied notebook has no completed training
execution recorded; do not treat the explanations below as evidence of a successful run.


## 1 Setup and experimental settings

All experimental choices are defined in the next cell. The primary settings are a monthly exposure cap of **1.5**,
transaction costs of **10 basis points per unit traded**, risk aversion **gamma 5**, and a **21 trading day** forecast approximation.
`TARGET_VOL_RATIO = 1.0` asks the historical gross timed portfolio to match the asset's volatility when calibrating c.
It does not guarantee that future net volatility will match it.

The default smoke run uses **Mkt, one seed, the year 1998, 20 GJR optimizer iterations, 10 neural accuracy steps and 5
decision steps**. A full run uses all six factors, five seeds and annual forecast years 1975 through 2025. These are
separate configurations with separate output folders.

`ACCURACY_OBJECTIVE = 'daily'` is the primary statistical benchmark. Setting it to `'monthly'` runs a distinct control
with a monthly variance loss. `EXTRA_ACCURACY_CONTROL = True` adds continued statistical training after A, using the
same maximum number of optimizer steps as B; this does not imply equal computational cost.

For a no-leverage experiment with `CAP = 1.0`, choose a lower volatility target, such as `TARGET_VOL_RATIO = 0.69`.
Matching the full asset volatility under a cap of one pushes this calibration toward full investment. The code rejects
a target ratio at or above the cap rather than silently treating that saturation as a useful timing experiment.

Freeze the primary choices before inspecting new test outcomes. Existing forecasts are preserved: revised outputs use
`data/LSTM-Forecasts-revised-smoke/` or `data/LSTM-Forecasts-revised/`, followed by a run identifier.


In [ ]:
import copy
import hashlib
import json
import math
import os
import platform
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy.optimize import brentq
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

DATA = Path('data')
SMOKE_TEST = False
ALL_FACTORS = ['Mkt', 'SMB', 'HML', 'Mom', 'RMW', 'CMA']
FACTOR_MAP = {'Mkt': 'Mkt-RF', 'SMB': 'SMB', 'HML': 'HML', 'Mom': 'Mom', 'RMW': 'RMW', 'CMA': 'CMA'}
FACTORS = ['Mkt'] if SMOKE_TEST else ALL_FACTORS
N_SEEDS = 1 if SMOKE_TEST else 5
FIRST_YEAR = 1998 if SMOKE_TEST else 1975
LAST_YEAR = 1998 if SMOKE_TEST else 2025
VAL_YEARS = 5
MIN_TRAIN_DAYS = 1260
MIN_CAL_MONTHS = 24
K_DAYS = 21
ZHAO_LEN = 126
CAP = 1.5
TARGET_VOL_RATIO = 1.0
GAMMA = 5.0
KAPPA = 0.001
BORROW_SPREAD_ANNUAL = 0.0  # nonzero spread is an explicit robustness experiment
ACCURACY_OBJECTIVE = 'daily'  # alternative control: 'monthly'
EXTRA_ACCURACY_CONTROL = False
GJR_STEPS = 20 if SMOKE_TEST else 200
A_STEPS = 10 if SMOKE_TEST else 400
B_STEPS = 5 if SMOKE_TEST else 300
A_LR, B_LR = 3e-3, 3e-3
BASE_SEED = 2026
PROTOCOL_VERSION = 'clean-validation-refiltered-annual-c-v1'
OUT_BASE = DATA / ('LSTM-Forecasts-revised-smoke' if SMOKE_TEST else 'LSTM-Forecasts-revised')

assert ACCURACY_OBJECTIVE in ('daily', 'monthly')
assert 0 < TARGET_VOL_RATIO < CAP, 'Choose a target below the cap; cap=1, target=1 saturates at buy and hold.'

CONFIG = {key: globals()[key] for key in [
    'SMOKE_TEST', 'FACTORS', 'N_SEEDS', 'FIRST_YEAR', 'LAST_YEAR', 'VAL_YEARS', 'MIN_TRAIN_DAYS',
    'MIN_CAL_MONTHS', 'K_DAYS', 'ZHAO_LEN', 'CAP', 'TARGET_VOL_RATIO', 'GAMMA', 'KAPPA',
    'BORROW_SPREAD_ANNUAL', 'ACCURACY_OBJECTIVE', 'EXTRA_ACCURACY_CONTROL', 'GJR_STEPS',
    'A_STEPS', 'B_STEPS', 'A_LR', 'B_LR', 'BASE_SEED', 'PROTOCOL_VERSION']}
CONFIG['factor_map'] = FACTOR_MAP
print('SMOKE RUN: no conclusions' if SMOKE_TEST else 'Full experiment')


## 2 Common input data and realized variance

The loader reads the same core CSV family as the final analysis:

- `data/ff_daily_garchdata_long.csv` for daily excess returns;
- `data/ff_monthly_baselinedata.csv` for official monthly excess returns;
- `data/ff_monthly_rf.csv` for the monthly risk-free return.

The factor mapping is `Mkt` to `Mkt-RF`, with SMB, HML, momentum, RMW and CMA taken from the core exports. This replaces
the earlier training notebook's separate FF3-based loading. Daily dates are parsed explicitly as day-month-year. Keys,
finite values, calendar continuity and forecast-origin alignment are checked.

Monthly realized variance is reconstructed as

$$RV_m = \sum_{d\in m}r_d^2.$$

The monthly mean is not subtracted. This convention is consistent with the zero-mean model used here. RV is an outcome
used for training labels and forecast evaluation; the RV of a test month is not an input available at its forecast origin.

Small differences between reconstructed RV and stored CSV RV are reported. New accuracy comparisons should use the
reconstructed RV exported by this run, rather than mixing it with older rounded or demeaned targets. Input file hashes
are saved with each experiment identity.

The primary history begins in 1963 for all six factors. A comparison against 1926-onward history would require a separate
matched-data experiment; differences from the old notebook cannot be attributed solely to the training objective.


In [ ]:
def load_core_data(data_dir):
    paths = {name: data_dir / name for name in [
        'ff_daily_garchdata_long.csv', 'ff_monthly_baselinedata.csv', 'ff_monthly_rf.csv']}
    for path in paths.values():
        if not path.exists():
            raise FileNotFoundError(f'Put this notebook beside data/. Missing: {path}')
    d = pd.read_csv(paths['ff_daily_garchdata_long.csv'])
    m = pd.read_csv(paths['ff_monthly_baselinedata.csv'])
    rf_table = pd.read_csv(paths['ff_monthly_rf.csv'], index_col='month')
    d['date'] = pd.to_datetime(d['Date'], format='%d-%m-%Y')
    assert not d.duplicated(['date', 'series']).any()
    assert not m.duplicated(['month', 'series']).any()
    assert rf_table.index.is_unique
    names = list(FACTOR_MAP.values())
    dw = d.pivot(index='date', columns='series', values='ret').sort_index()[names]
    mw = m.pivot(index='month', columns='series', values='ret_m').sort_index()[names]
    stored_rv = m.pivot(index='month', columns='series', values='rv').sort_index()[names]
    mw.index = pd.PeriodIndex(mw.index, freq='M')
    stored_rv.index = pd.PeriodIndex(stored_rv.index, freq='M')
    rf_table.index = pd.PeriodIndex(rf_table.index, freq='M')
    assert np.isfinite(dw.to_numpy()).all() and np.isfinite(mw.to_numpy()).all()
    assert mw.index.equals(pd.period_range(mw.index.min(), mw.index.max(), freq='M', name=mw.index.name))
    reconstructed = dw.pow(2).groupby(dw.index.to_period('M')).sum(min_count=1)
    assert (reconstructed > 0).all().all()
    print('Maximum stored-versus-reconstructed RV difference:',
          float((stored_rv - reconstructed.reindex(stored_rv.index)).abs().max().max()))
    rename = {value: key for key, value in FACTOR_MAP.items()}
    dw, mw, reconstructed = [x.rename(columns=rename) for x in (dw, mw, reconstructed)]
    mw['RF'] = rf_table['RF'].reindex(mw.index)
    assert mw['RF'].notna().all()
    hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in paths.items()}
    return dw, mw, reconstructed, hashes


def factor_months(fac):
    d = daily[fac].dropna()
    end = pd.Series(np.arange(len(d)), index=d.index).groupby(d.index.to_period('M')).max()
    table = pd.DataFrame({'origin': end.shift(1), 'f': monthly[fac].reindex(end.index),
                          'rf': monthly['RF'].reindex(end.index), 'rv': rv_monthly[fac].reindex(end.index)})
    table = table.dropna()
    table['origin'] = table['origin'].astype(int)
    table = table.loc[table['origin'] >= ZHAO_LEN - 1].copy()
    table['year'] = table.index.year
    assert table.index.is_unique and np.isfinite(table.to_numpy()).all()
    origin_months = d.index[table['origin'].to_numpy()].to_period('M')
    assert (origin_months == table.index - 1).all()
    return d, table


def split_history(dates, test_year):
    test_start = pd.Timestamp(f'{test_year}-01-01')
    val_start = pd.Timestamp(f'{test_year - VAL_YEARS}-01-01')
    n_train, n_history = int((dates < val_start).sum()), int((dates < test_start).sum())
    if n_train < MIN_TRAIN_DAYS or n_history <= n_train:
        raise ValueError('Insufficient earlier data for training and validation.')
    return n_train, n_history, val_start


daily, monthly, rv_monthly, DATA_HASHES = load_core_data(DATA)
print(daily.index.min(), 'through', daily.index.max())


## 3 GJR and GARCH LSTM variance models

The conditional mean is fixed at zero. The GJR variance update is

$$h_t = \omega + (\alpha + \gamma\,1[r_{t-1}<0])r_{t-1}^2 + \beta h_{t-1}.$$

The parameter transformations enforce positive variance contributions and persistence
\(P=\alpha+\beta+\gamma/2<0.9995\). The asymmetric coefficient may be negative provided the positive- and negative-shock
contributions remain nonnegative. The starting variance `var0` is calculated from the fitting prefix only, before validation.
The GJR path is computed by FFT convolution; Section 7 checks it against a direct recursion.

The neural model retains the existing 126-day architecture: its gates read the shock and variance, and its output scales
the GJR update. The neural output weights start at zero, so initialization reproduces the GJR path before training.

For next month's variance, both objectives use the same `monthly_forecast()` transformation:

$$\widehat V_m = K\bar h + (H_{T+1}-\bar h)\frac{1-P^K}{1-P},\qquad
\bar h=\frac{\omega}{1-P},\quad K=21.$$

Here T is the preceding month-end trading day. For the neural model this is a **GJR mean-reversion approximation** based
on its next-day forecast; it does not simulate the neural state through all future days. Twenty-one sessions also
approximate a calendar month. Keep these assumptions explicit when interpreting accuracy.

Use the GJR forecasts exported here when comparing this implementation with LSTM. A separately fitted GJR model with
different mean assumptions, parameter constraints or estimation dates is a distinct benchmark.


In [ ]:
def logit(p):
    return math.log(p / (1 - p))


class VolModel(nn.Module):
    def __init__(self, kind, var0):
        super().__init__()
        assert kind in ("gjr", "zhao")
        self.kind = kind
        self.var0 = float(var0)         # training-sample daily variance: sets the scale and the starting values
        # free parameters, starting from typical daily equity values (P = 0.98, beta = 0.90)
        self.u_mu = nn.Parameter(torch.tensor(0.0), requires_grad=False)  # mean fixed at zero
        self.u_P = nn.Parameter(torch.tensor(logit(0.98 / 0.9995)))
        self.u_b = nn.Parameter(torch.tensor(logit(0.90 / 0.98)))
        self.u_a = nn.Parameter(torch.tensor(logit(0.35)))
        self.u_w = nn.Parameter(torch.tensor(0.0))
        if kind == "zhao":
            self.H = 8                                      # LSTM hidden size
            self.gates = nn.Linear(2, 3 * self.H)           # forget, input and cell gates, fed (shock, variance)
            self.u_out = nn.Parameter(torch.zeros(self.H))  # w in sigma2 = o (1 + w . tanh(c)); zero = pure GJR

    def params(self):
        P = 0.9995 * torch.sigmoid(self.u_P)
        beta = P * torch.sigmoid(self.u_b)
        rest = P - beta                       # = alpha + gamma / 2
        s = torch.sigmoid(self.u_a)
        alpha = 2 * rest * s                  # weight on good-news shocks
        gam = 2 * rest * (1 - s) - alpha      # extra weight on bad-news shocks (can be negative)
        omega = self.var0 * (1 - P) * torch.exp(self.u_w)
        return dict(mu=self.u_mu * 1e-3, P=P, alpha=alpha, beta=beta, gam=gam, omega=omega)

    def gjr_path(self, r):
        """Daily variance h (for each day, using data up to the day before) and h_next (forecast for the next day)."""
        p = self.params()
        eps = r - p["mu"]
        x = p["omega"] + (p["alpha"] + p["gam"] * (eps < 0).to(r.dtype)) * eps ** 2   # x[s] drives h[s + 1]
        n = r.shape[0]
        kern = torch.exp(torch.arange(n, dtype=r.dtype) * torch.log(p["beta"]))      # beta ** j
        L = 1 << (2 * n - 1).bit_length()
        conv = torch.fft.irfft(torch.fft.rfft(x, L) * torch.fft.rfft(kern, L), L)[:n]
        h0 = torch.tensor(self.var0, dtype=r.dtype)
        h_next = conv + kern * p["beta"] * h0
        h = torch.cat([h0.reshape(1), h_next[:-1]])
        return eps, h, h_next, p

    def zhao_windows(self, r, ends, eps, h, p):
        """Runs the GARCH-LSTM over the 126 days ending at each index in `ends`. Each window starts from the GJR
        variance of its first day with an empty cell. Column k is the forecast for day start + k + 1."""
        L, H, f32 = ZHAO_LEN, self.H, torch.float32
        starts = ends - L + 1
        # float32 on percent units (x100 returns, x1e4 variances) for speed; matches float64 to about 3e-7
        s2 = (1e4 * h[starts]).to(f32)
        E = (100 * eps[starts.unsqueeze(1) + torch.arange(L).unsqueeze(0)]).to(f32)
        om, al, ga, be = [(p[k] * (1e4 if k == "omega" else 1)).to(f32) for k in ("omega", "alpha", "gam", "beta")]
        c = torch.zeros(len(ends), H, dtype=f32)
        u = self.u_out.to(f32)
        w = 0.99 * u / (1 + u.abs().sum())     # keeps sum |w| < 1, so the variance stays positive
        Wg, bg = self.gates.weight.to(f32), self.gates.bias.to(f32)
        preds = []
        for k in range(L):
            e = E[:, k]
            o = om + (al + ga * (e < 0).to(f32)) * e * e + be * s2           # the GJR update
            g = torch.addmm(bg, torch.stack([e, s2], 1), Wg.t())
            c = torch.sigmoid(g[:, :H]) * c + torch.sigmoid(g[:, H:2 * H]) * torch.tanh(g[:, 2 * H:])
            s2 = o * (1 + torch.tanh(c) @ w)
            preds.append(s2)
        return torch.stack(preds, 1).to(torch.float64) / 1e4

    def next_var(self, r, idx):
        """One-day-ahead variance forecast made at the close of each day in idx."""
        eps, h, h_next, p = self.gjr_path(r)
        Hn = h_next[idx]
        if self.kind == "zhao":
            ok = idx >= ZHAO_LEN - 1          # the first half-year of data has no full window: plain GJR there
            if ok.any():
                Hn = Hn.clone()
                Hn[ok] = self.zhao_windows(r, idx[ok], eps, h, p)[:, -1]
        return Hn, eps, p

    def monthly_forecast(self, r, me_idx):
        """Variance forecast for next month, made on the month-end days me_idx.
        The model's one-day-ahead variance is carried 21 days forward with the GJR mean-reversion formula. For the
        GARCH-LSTM this is an approximation: it does not run the network forward over the month."""
        Hn, _, p = self.next_var(r, me_idx)
        P = p["P"]
        hbar = p["omega"] / (1 - P)
        return K_DAYS * hbar + (Hn - hbar) * (1 - P ** K_DAYS) / (1 - P)


## 4 Statistical loss decision loss and allocation policy

**Accuracy objective A.** The primary loss is the mean daily Gaussian quasi-likelihood,

$$L_A=E\left[\log H_t+\frac{r_t^2}{H_t}\right].$$

The optional monthly-accuracy control instead minimizes QLIKE against monthly RV using the same monthly forecast function.
It is a separately identified experiment, not a replacement selected after seeing which gives better test performance.

**Decision objective B.** Exposure is

$$w_m=\min(c_m/\widehat V_m,\mathrm{CAP}).$$

After transaction and any configured borrowing costs, B maximizes

$$J_B=E[R_m^{net}]-\frac{\gamma}{2}E[(R_m^{net})^2].$$

This second-moment objective is the primary decision metric. It is not identical to mean minus gamma over two times
the variance. Monthly objective values are multiplied by 1200 only when reported as annual percentage points.

**Annual calibration of c.** At each January boundary, calibrate c on the current model's variance path and asset returns
from strictly earlier months, after at least 24 calibration months. The historical gross timed portfolio targets
`TARGET_VOL_RATIO` times the asset's volatility. Root-finding is bounded; the gradient with respect to forecasts uses
implicit differentiation. Invalid inputs or an unstable implicit gradient raise an error.

During fitting, this annual rule is replayed on the fitted historical model path. At deployment, the newly fitted model
re-filters pre-year history to calibrate c, and that c is held fixed over the forward year. Historical replay is in sample
for the model parameters; only the exported later test year is out of sample. This policy differs from calibrating c on
stored forecasts produced by older models. The final analysis must use this run's exported weights to preserve the policy.

**Portfolio accounting.** Transaction cost is charged on the difference between the new target exposure and the previous
position after market drift. An optional annual borrowing spread applies to exposure above one. Cash and borrowing at
the risk-free rate are represented through excess returns; total wealth is formed by adding RF back later. The drift
formula retains the original approximation of ignoring the small previous transaction-cost effect on wealth.

The first position in the complete path is entered free. Costs are calculated over the continuous monthly path before
training and validation observations are separated, so validation's first rebalance is charged. This also preserves
turnover at annual boundaries when the exported strategy is evaluated.


In [ ]:
def qmle_loss(model, r, days):
    Hn, eps, _ = model.next_var(r, days - 1)
    return (torch.log(Hn) + eps[days].square() / Hn).mean()


def qmle_window_loss(model, r, ends):
    eps, h, _, p = model.gjr_path(r)
    pred = model.zhao_windows(r, ends, eps, h, p)
    targets = (ends - ZHAO_LEN + 2).unsqueeze(1) + torch.arange(ZHAO_LEN).unsqueeze(0)
    return (torch.log(pred) + eps[targets].square() / pred).mean()


def qlike_torch(V, rv):
    x = rv / V
    return (x - torch.log(x) - 1).mean()


def calibrate_c_numpy(V, f, ratio=TARGET_VOL_RATIO, cap=CAP):
    V, f = np.asarray(V, float), np.asarray(f, float)
    if len(V) < MIN_CAL_MONTHS or V.shape != f.shape:
        raise ValueError('Calibration needs aligned monthly histories of sufficient length.')
    if not np.isfinite(V).all() or not np.isfinite(f).all() or (V <= 0).any():
        raise ValueError('Invalid calibration input.')
    if not 0 < ratio < cap or f.std(ddof=1) <= 0:
        raise ValueError('Target must be positive, below the cap, and asset volatility must be positive.')
    target = ratio * f.std(ddof=1)
    gap = lambda c: (np.clip(c / V, 0, cap) * f).std(ddof=1) - target
    # At zero exposure the gap is negative; at all-capped exposure it is positive.
    hi = cap * V.max() * (1 + 1e-8)
    root = brentq(gap, 0.0, hi, xtol=1e-16, rtol=1e-12)
    if abs(gap(root)) > 1e-8 * max(target, 1e-8):
        raise RuntimeError('Calibration failed its residual check.')
    return root


def calibrate_c_torch(V, f):
    root = calibrate_c_numpy(V.detach().cpu().numpy(), f.detach().cpu().numpy())
    c0 = V.new_tensor(root)
    if not torch.is_grad_enabled() or not V.requires_grad:
        return c0
    fd = f.detach()
    with torch.enable_grad():
        c_probe = c0.detach().requires_grad_(True)
        gap = (torch.clamp(c_probe / V.detach(), 0, CAP) * fd).std() - TARGET_VOL_RATIO * fd.std()
        gc = torch.autograd.grad(gap, c_probe)[0].detach()
    if not torch.isfinite(gc) or float(gc) <= 1e-12:
        raise RuntimeError('Unstable implicit calibration gradient; inspect saturation and inputs.')
    gV = (torch.clamp(c0 / V, 0, CAP) * fd).std() - TARGET_VOL_RATIO * fd.std()
    return c0 - (gV - gV.detach()) / gc


def annual_c_torch(V, f, years):
    years = np.asarray(years)
    values = [V.new_tensor(float('nan')) for _ in range(len(V))]
    for year in np.unique(years):
        past = np.flatnonzero(years < year)
        if len(past) < MIN_CAL_MONTHS:
            continue
        c = calibrate_c_torch(V[past], f[past])
        for j in np.flatnonzero(years == year):
            values[j] = c
    return torch.stack(values)


def portfolio_net_torch(w, f, rf, initial_drift=None):
    first = w[0] if initial_drift is None else w.new_tensor(initial_drift)
    denominator = 1 + rf[:-1] + w[:-1] * f[:-1]
    if not bool((denominator > 0).all()):
        raise ValueError('Nonpositive gross portfolio wealth encountered.')
    drift = torch.cat([first.reshape(1), w[:-1] * (1 + f[:-1] + rf[:-1]) / denominator])
    turnover = (w - drift).abs()
    borrowing = torch.clamp(w - 1, min=0) * BORROW_SPREAD_ANNUAL / 12
    return w * f - KAPPA * turnover - borrowing, turnover


def portfolio_net_numpy(w, f, rf, initial_drift=None):
    w, f, rf = [np.asarray(x, float) for x in (w, f, rf)]
    assert len(w) and w.shape == f.shape == rf.shape
    assert np.isfinite(np.column_stack([w, f, rf])).all()
    denominator = 1 + rf[:-1] + w[:-1] * f[:-1]
    if not (denominator > 0).all():
        raise ValueError('Nonpositive gross portfolio wealth encountered.')
    first = w[0] if initial_drift is None else initial_drift
    drift = np.r_[first, w[:-1] * (1 + f[:-1] + rf[:-1]) / denominator]
    turnover = np.abs(w - drift)
    borrowing = np.maximum(w - 1, 0) * BORROW_SPREAD_ANNUAL / 12
    return w * f - KAPPA * turnover - borrowing, turnover


def decision_objective(net):
    return net.mean() - 0.5 * GAMMA * net.square().mean()


def tensor_months(table):
    return (torch.tensor(table['origin'].to_numpy(), dtype=torch.long),
            torch.tensor(table['f'].to_numpy()), torch.tensor(table['rf'].to_numpy()),
            torch.tensor(table['rv'].to_numpy()))


def decision_path(model, r, table):
    origins, f, rf, _ = tensor_months(table)
    V = model.monthly_forecast(r, origins)
    if not bool(torch.isfinite(V).all() and (V > 0).all()):
        raise FloatingPointError('Invalid monthly forecast.')
    c = annual_c_torch(V, f, table['year'].to_numpy())
    valid = torch.isfinite(c)
    w = torch.clamp(c[valid] / V[valid], 0, CAP)
    net, turnover = portfolio_net_torch(w, f[valid], rf[valid])
    return net, w, turnover, valid
    

## 5 Training validation and checkpoint selection

For a forecast year Y, parameter fitting ends before **1 January Y minus five years**. The following five calendar years
form the internal validation block; all of them precede the test year. For example, forecasting 1998 fits parameters
on observations before 1993, validates on 1993 through 1997, and exports forecasts for 1998.

The procedure is:

1. Calculate `var0` and fit statistical GJR using only the fitting prefix.
2. Initialize each neural seed from that GJR fit. Train A on fitting data and select its checkpoint on validation loss.
3. Copy the selected A model into B. Train B on fitting utility and select its checkpoint on validation utility.
4. Keep the selected checkpoint and use it to forecast the later test year.

Every year starts freshly. No previous-year neural checkpoint is reused, avoiding the possibility that a warm start has
already fitted observations in the new validation block. B retains the A solution if its subsequent optimization never
improves validation utility. Finite-loss and gradient checks stop numerical failures instead of exporting bad forecasts.

Validation observations influence checkpoint selection, as intended for tuning. Their earlier returns can update later
volatility states and c calibration; they do not enter gradient-based parameter fitting. Parameters are **not refit on
validation after selection** in this version. A full-history refit would require a separately specified procedure for
selecting training duration and refitting A and B.

Training and validation losses, selected steps, exposure, cap frequency and turnover are recorded. They help distinguish
optimization failure, unstable training and weak generalization. They do not by themselves establish overfitting or
successful out-of-sample performance.


In [ ]:
GJR_KEYS = ['u_P', 'u_b', 'u_a', 'u_w']


def network_params(model):
    return list(model.gates.parameters()) + [model.u_out] if model.kind == 'zhao' else []


def ensure_finite(loss, label):
    if not bool(torch.isfinite(loss)):
        raise FloatingPointError(f'Nonfinite {label}; stop instead of exporting bad forecasts.')


def fit_A_gjr(model, r_fit, table_fit):
    days = torch.arange(1, len(r_fit))
    origins, _, _, rv = tensor_months(table_fit)
    def loss_fn():
        return (qmle_loss(model, r_fit, days) if ACCURACY_OBJECTIVE == 'daily'
                else qlike_torch(model.monthly_forecast(r_fit, origins), rv))
    initial = float(loss_fn().detach())
    opt = torch.optim.LBFGS([getattr(model, key) for key in GJR_KEYS], max_iter=GJR_STEPS,
                            tolerance_grad=1e-10, tolerance_change=1e-13, line_search_fn='strong_wolfe')
    def closure():
        opt.zero_grad()
        loss = loss_fn()
        ensure_finite(loss, 'GJR fitting loss')
        loss.backward()
        return loss
    opt.step(closure)
    final = loss_fn()
    ensure_finite(final, 'GJR final loss')
    return [{'step': GJR_STEPS, 'train_loss': float(final.detach()), 'initial_loss': initial}]


def fit_A_zhao(model, r_history, n_train, table, val_start, seed, steps=A_STEPS):
    gen = torch.Generator().manual_seed(seed)
    L = ZHAO_LEN
    # Target days in each validation window all lie on/after n_train.
    val_ends = torch.arange(n_train + L - 2, len(r_history) - 1, L)
    if not len(val_ends):
        raise ValueError('Validation block is too short.')
    train_mask = table.index.to_timestamp() < val_start
    origins, _, _, rv = tensor_months(table)
    opt = torch.optim.Adam([{'params': [getattr(model, key) for key in GJR_KEYS], 'lr': A_LR / 10},
                            {'params': network_params(model), 'lr': A_LR}])
    def validation():
        with torch.no_grad():
            return (qmle_window_loss(model, r_history, val_ends) if ACCURACY_OBJECTIVE == 'daily'
                    else qlike_torch(model.monthly_forecast(r_history, origins[~train_mask]), rv[~train_mask]))
    best = float(validation())
    ensure_finite(torch.tensor(best), 'initial accuracy validation loss')
    best_state, best_step, since = copy.deepcopy(model.state_dict()), 0, 0
    logs = [{'step': 0, 'val_loss': best}]
    for step in range(1, steps + 1):
        opt.zero_grad()
        if ACCURACY_OBJECTIVE == 'daily':
            ends = torch.randint(L - 1, n_train - 1, (64,), generator=gen)
            loss = qmle_window_loss(model, r_history[:n_train], ends)
        else:
            loss = qlike_torch(model.monthly_forecast(r_history[:n_train], origins[train_mask]), rv[train_mask])
        ensure_finite(loss, 'accuracy training loss')
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
        opt.step()
        if step % 10 == 0 or step == steps:
            val = validation()
            ensure_finite(val, 'accuracy validation loss')
            value = float(val)
            logs.append({'step': step, 'train_loss': float(loss.detach()), 'val_loss': value})
            if value < best - 1e-6:
                best, best_state, best_step, since = value, copy.deepcopy(model.state_dict()), step, 0
            else:
                since += 1
                if since >= 6:
                    break
    model.load_state_dict(best_state)
    logs.append({'selected_step': best_step, 'selected_val_loss': best})
    return logs


def fit_B(model, r_history, table, val_start, lr=B_LR):
    train_rows = table.index.to_timestamp() < val_start
    opt = torch.optim.Adam([{'params': [getattr(model, key) for key in GJR_KEYS], 'lr': lr}] +
                           ([{'params': network_params(model), 'lr': lr}] if model.kind == 'zhao' else []))
    best, best_state, best_step, since, logs = -np.inf, None, 0, 0, []
    for step in range(B_STEPS + 1):
        opt.zero_grad()
        net, w, turnover, valid = decision_path(model, r_history, table)
        training = torch.tensor(train_rows)[valid]
        if int(training.sum()) < 12 or int((~training).sum()) < 12:
            raise ValueError('Need at least 12 eligible training and validation months.')
        train_u, val_u = decision_objective(net[training]), decision_objective(net[~training])
        ensure_finite(train_u, 'decision training objective')
        ensure_finite(val_u, 'decision validation objective')
        value = float(val_u.detach())
        logs.append({'step': step, 'train_utility': float(train_u.detach()), 'val_utility': value,
                     'mean_weight': float(w.detach().mean()), 'at_cap': float((w.detach() >= CAP - 1e-9).double().mean()),
                     'turnover_per_month': float(turnover.detach().mean())})
        if value > best + 1e-7:
            best, best_state, best_step, since = value, copy.deepcopy(model.state_dict()), step, 0
        else:
            since += 1
        if step == B_STEPS or since >= 30:
            break
        (-100 * train_u).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
        opt.step()
    model.load_state_dict(best_state)
    logs.append({'selected_step': best_step, 'selected_val_utility': best})
    return logs


## 6 Annual forecasting ensembles and saved outputs

For each factor and forecast year, the loop builds the fitting and validation blocks, trains GJR A/B and neural A/B
models, and then exports the forward year. Each monthly forecast receives a literal daily-data prefix ending at the
preceding month-end trading day. Test-month returns and RV are retained as outcomes for later scoring.

The primary neural ensemble averages **variance forecasts before allocation**. Its c is calibrated on the corresponding
averaged historical forecasts. Applying the inverse-variance rule to this average is generally different from averaging
individually calculated seed weights. Use this ensemble convention consistently in all analysis sections. B seeds are
trained independently; they are not jointly optimized for ensemble utility.

Saved columns include:

| Prefix or field | Meaning |
|---|---|
| `f`, `rf`, `rv` | Monthly excess return, risk-free return, and reconstructed realized variance |
| `origin_date` | Last daily observation supplied for the forecast |
| `parameter_training_before` | Boundary before which parameter-fitting observations lie |
| `validation_before` | Start of the later test year |
| `V_...` | Monthly variance forecast, including individual seeds and ensembles |
| `C_...` | Annual calibration constant applied to that forecast |
| `W_...` | Exposure generated by this run's calibration and allocation rule |

Each completed factor-year saves a CSV and a checkpoint containing model states and diagnostics. A run identifier
covers settings, input hashes, implementation fingerprints and dependency versions. Completed years with both files
can be resumed within the same experiment identity. Changing the configuration creates a different identity.

Full runs write consolidated factor files under `data/LSTM-Forecasts-revised/<run-id>/`; smoke runs use the separate
smoke folder. The original `data/LSTM-Forecasts/` files are not overwritten. Old results should not be expected to match:
the history, validation, initialization and calibration protocol have changed.


In [ ]:
def atomic_csv(frame, path):
    temp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temp, index_label='month', float_format='%.12g')
    os.replace(temp, path)


def experiment_folder():
    import marshal
    import types
    def stable_code(obj):
        # Jupyter changes cell filenames/absolute line numbers between kernels.
        constants = tuple(stable_code(c) if isinstance(c, types.CodeType) else c for c in obj.co_consts)
        return obj.replace(co_filename='', co_firstlineno=1, co_linetable=b'', co_consts=constants)
    functions = [load_core_data, factor_months, split_history, calibrate_c_numpy, calibrate_c_torch,
                 annual_c_torch, portfolio_net_torch, portfolio_net_numpy, decision_objective, tensor_months,
                 decision_path, qmle_loss, qmle_window_loss, qlike_torch, fit_A_gjr, fit_A_zhao, fit_B,
                 network_params, ensure_finite, logit, atomic_csv, export_year, run_factor]
    methods = [getattr(VolModel, name) for name in ['__init__', 'params', 'gjr_path', 'zhao_windows', 'next_var', 'monthly_forecast']]
    signature = hashlib.sha256(b''.join(marshal.dumps(stable_code(f.__code__)) for f in functions + methods)).hexdigest()
    manifest = {'config': CONFIG, 'data_sha256': DATA_HASHES, 'implementation_sha256': signature,
                'versions': {'python': platform.python_version(), 'numpy': np.__version__,
                             'pandas': pd.__version__, 'scipy': scipy.__version__, 'torch': torch.__version__}}
    run_id = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:12]
    folder = OUT_BASE / run_id
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / 'manifest.json'
    if path.exists():
        assert json.loads(path.read_text()) == manifest, 'Run identity mismatch.'
    else:
        path.write_text(json.dumps(manifest, indent=2))
    return folder, run_id


def export_year(models, d, table, year, val_start):
    history = table.loc[table['year'] < year]
    test = table.loc[table['year'] == year]
    if len(history) < MIN_CAL_MONTHS or len(test) != 12:
        raise ValueError('A year needs 12 test months and sufficient calibration history.')
    r_all = torch.tensor(d.to_numpy())
    r_history = r_all[:int((d.index < pd.Timestamp(f'{year}-01-01')).sum())]
    h_origins, h_f, _, _ = tensor_months(history)
    out = test[['f', 'rf', 'rv']].copy()
    out['origin_date'] = [str(d.index[i].date()) for i in test['origin']]
    out['parameter_training_before'] = str(val_start.date())
    out['validation_before'] = f'{year}-01-01'
    historical, future = {}, {}
    with torch.no_grad():
        for name, model in models.items():
            hist = model.monthly_forecast(r_history, h_origins)
            pred = torch.stack([model.monthly_forecast(r_all[:origin + 1], torch.tensor([origin]))[0]
                                for origin in test['origin'].to_numpy()])
            historical[name], future[name] = hist, pred
        for obj in ['A', 'B'] + (['Aextra'] if EXTRA_ACCURACY_CONTROL else []):
            names = [f'zhao_{obj}_s{s}' for s in range(N_SEEDS)]
            historical[f'zhao_{obj}_ensemble'] = torch.stack([historical[n] for n in names]).mean(0)
            future[f'zhao_{obj}_ensemble'] = torch.stack([future[n] for n in names]).mean(0)
        for name, pred in future.items():
            c = calibrate_c_torch(historical[name], h_f)
            weights = torch.clamp(c / pred, 0, CAP)
            out[f'V_{name}'] = pred.cpu().numpy()
            out[f'C_{name}'] = float(c)
            out[f'W_{name}'] = weights.cpu().numpy()
    numeric = out.select_dtypes('number')
    assert np.isfinite(numeric.to_numpy()).all()
    assert (out.filter(regex='^V_') > 0).all().all()
    assert ((out.filter(regex='^W_') >= 0) & (out.filter(regex='^W_') <= CAP)).all().all()
    return out


def run_factor(fac, folder, run_id):
    d, table = factor_months(fac)
    pieces = []
    for year in range(FIRST_YEAR, LAST_YEAR + 1):
        year_dir = folder / fac
        year_dir.mkdir(exist_ok=True)
        csv_path, checkpoint = year_dir / f'{year}.csv', year_dir / f'{year}.pt'
        if csv_path.exists() and checkpoint.exists():
            saved = pd.read_csv(csv_path, index_col='month')
            assert len(saved) == 12 and saved.index.tolist() == [str(x) for x in pd.period_range(f'{year}-01', f'{year}-12', freq='M')]
            pieces.append(saved)
            print(f'{fac} {year}: resumed')
            continue
        t0 = time.time()
        n_train, n_history, val_start = split_history(d.index, year)
        r_history = torch.tensor(d.iloc[:n_history].to_numpy())
        r_fit = r_history[:n_train]
        hist_table = table.loc[table['year'] < year]
        fit_table = hist_table.loc[hist_table.index.to_timestamp() < val_start]
        var0 = float(r_fit.var())
        models, logs = {}, {}
        gjr = VolModel('gjr', var0)
        logs['gjr_A'] = fit_A_gjr(gjr, r_fit, fit_table)
        models['gjr_A'] = gjr
        gjr_B = copy.deepcopy(gjr)
        logs['gjr_B'] = fit_B(gjr_B, r_history, hist_table, val_start, lr=1e-2)
        models['gjr_B'] = gjr_B
        for seed_index in range(N_SEEDS):
            seed = BASE_SEED + 1000 * ALL_FACTORS.index(fac) + 100 * seed_index + year
            torch.manual_seed(seed)
            A = VolModel('zhao', var0)
            A.load_state_dict({**A.state_dict(), **gjr.state_dict()})
            logs[f'zhao_A_s{seed_index}'] = fit_A_zhao(A, r_history, n_train, hist_table, val_start, seed)
            models[f'zhao_A_s{seed_index}'] = A
            B = copy.deepcopy(A)
            logs[f'zhao_B_s{seed_index}'] = fit_B(B, r_history, hist_table, val_start)
            models[f'zhao_B_s{seed_index}'] = B
            if EXTRA_ACCURACY_CONTROL:
                extra = copy.deepcopy(A)
                logs[f'zhao_Aextra_s{seed_index}'] = fit_A_zhao(extra, r_history, n_train, hist_table, val_start,
                                                             seed + 100000, steps=B_STEPS)
                models[f'zhao_Aextra_s{seed_index}'] = extra
        out = export_year(models, d, table, year, val_start)
        payload = {'run_id': run_id, 'factor': fac, 'year': year, 'var0': var0,
                   'parameter_training_before': str(val_start.date()), 'logs': logs,
                   'state_dicts': {name: model.state_dict() for name, model in models.items()}}
        temp = checkpoint.with_suffix('.pt.tmp')
        torch.save(payload, temp)
        os.replace(temp, checkpoint)
        atomic_csv(out, csv_path)
        pieces.append(out)
        print(f'{fac} {year}: {time.time() - t0:.1f}s', flush=True)
    combined = pd.concat(pieces)
    combined.index = pd.PeriodIndex(combined.index, freq='M').astype(str)
    assert combined.index.is_unique
    assert combined.index.tolist() == [str(x) for x in pd.period_range(combined.index[0], combined.index[-1], freq='M')]
    atomic_csv(combined, folder / f'{fac}_forecasts.csv')
    return combined


## 7 Run checks and the smoke experiment

The next cell first checks:

- agreement between Torch and NumPy portfolio returns and turnover;
- transaction-cost continuity across a split in the return history;
- calibration accuracy and an implicit gradient against finite differences on smooth synthetic data;
- invariance of an earlier forecast to changes in later returns;
- agreement between FFT and directly recursive GJR variances;
- fitting and validation dates preceding the test year.

These checks assess implementation behavior, not investment merit. The finite-difference case avoids cap boundaries,
where the function is not smooth. A successful check does not remove all numerical or modeling uncertainty.

With `SMOKE_TEST = True`, the cell then trains one factor, one seed and one forecast year with very few optimizer steps.
Confirm that it finishes, produces valid files and reports sensible diagnostics. **Do not interpret its Sharpe ratio or
utility as a research result.**

After the smoke experiment succeeds, change `SMOKE_TEST = False`, restart the kernel and run all cells. The full run is
substantially more expensive. If interrupted, rerun the same configuration to resume completed factor-years. There is
only one execution cell: do not reintroduce the obsolete `SAVED` loop or call `run_factor(fac)` without its run arguments.


In [ ]:
def run_checks():
    rng = np.random.default_rng(7)
    w, f, rf = rng.uniform(0, CAP, 60), rng.normal(0.005, 0.03, 60), np.full(60, 0.002)
    nt, tt = portfolio_net_torch(torch.tensor(w), torch.tensor(f), torch.tensor(rf))
    nn_, tn = portfolio_net_numpy(w, f, rf)
    np.testing.assert_allclose(nt.detach().numpy(), nn_, rtol=1e-12, atol=1e-14)
    np.testing.assert_allclose(tt.detach().numpy(), tn, rtol=1e-12, atol=1e-14)
    split = 35
    drift = w[split - 1] * (1 + f[split - 1] + rf[split - 1]) / (1 + rf[split - 1] + w[split - 1] * f[split - 1])
    carried, _ = portfolio_net_numpy(w[split:], f[split:], rf[split:], initial_drift=drift)
    np.testing.assert_allclose(carried, nn_[split:])

    V = torch.tensor(rng.uniform(0.002, 0.004, 100), requires_grad=True)
    ff = torch.tensor(rng.normal(0.005, 0.03, 100))
    c = calibrate_c_torch(V, ff)
    assert abs(float((torch.clamp(c / V, 0, CAP) * ff).std() / ff.std()) - TARGET_VOL_RATIO) < 1e-7
    grad = torch.autograd.grad(c, V)[0].detach().numpy()
    k, eps = 10, 1e-7
    plus, minus = V.detach().numpy().copy(), V.detach().numpy().copy()
    plus[k] += eps
    minus[k] -= eps
    fd = (calibrate_c_numpy(plus, ff.numpy()) - calibrate_c_numpy(minus, ff.numpy())) / (2 * eps)
    np.testing.assert_allclose(grad[k], fd, rtol=3e-3, atol=1e-6)

    torch.manual_seed(7)
    r = torch.tensor(rng.normal(0, 0.01, 400))
    for kind in ['gjr', 'zhao']:
        model = VolModel(kind, float(r[:200].var()))
        if kind == 'zhao':
            with torch.no_grad():
                model.u_out.fill_(0.03)  # exercise the neural branch, not just its zero-initialized GJR case
        origin = 249
        with torch.no_grad():
            original = model.monthly_forecast(r[:origin + 1], torch.tensor([origin]))
            altered = r.clone()
            altered[origin + 1:] *= 5
            same = model.monthly_forecast(altered, torch.tensor([origin]))
        torch.testing.assert_close(original, same, rtol=1e-5, atol=1e-10)
        if kind == 'gjr':
            eps_, h, _, p = model.gjr_path(r)
            direct = [r.new_tensor(model.var0)]
            for t in range(1, len(r)):
                e = eps_[t - 1]
                direct.append(p['omega'] + (p['alpha'] + p['gam'] * (e < 0)) * e.square() + p['beta'] * direct[-1])
            torch.testing.assert_close(h, torch.stack(direct), rtol=1e-9, atol=1e-12)

    fac = FACTORS[0]
    d, _ = factor_months(fac)
    n_train, n_history, val_start = split_history(d.index, FIRST_YEAR)
    assert d.index[n_train - 1] < val_start <= d.index[n_train]
    assert d.index[n_history - 1] < pd.Timestamp(f'{FIRST_YEAR}-01-01')
    print('Checks passed. Training and validation precede the test year.')


run_checks()
OUT, RUN_ID = experiment_folder()
print('Run:', RUN_ID, 'Outputs:', OUT)
results = {fac: run_factor(fac, OUT, RUN_ID) for fac in FACTORS}


## 8 Evaluate the exported allocation policy

The scorecard uses the exported `W_...` columns. Net returns are calculated over the full concatenated history before
the evaluation window is selected, preserving turnover across year and period boundaries. Buy and hold uses unit exposure.
Total-return wealth adds the monthly risk-free return, and maximum drawdown includes initial wealth of one.

The descriptive metrics are:

- **Exact decision objective** in annual percentage points, matching the second-moment objective used to train B;
- **Sharpe ratio**, annualized from monthly net excess returns;
- **Annualized volatility** of monthly net excess returns;
- **CAGR**, compounded from total returns rather than an annualized arithmetic mean;
- **Maximum drawdown**, measured from total-return wealth;
- **QLIKE**, evaluated on the variance forecast corresponding to the actual exported policy, including the primary ensemble.

The default evaluation window is 1998 through 2025. In a smoke run the scorecard covers only the available one-year
subset and is printed as smoke output. It is useful for checking plumbing only.

This cell does not perform hypothesis testing or multiple-testing correction. Rerun the paired utility and Sharpe
comparisons in the final analysis using the new net-return series and common calendar blocks across factors. Lead the
A/B decision-training comparison with the exact training objective. Report uncertainty rather than inferring superiority
from a point estimate. Retrospective equal-volatility rescaling remains a separate diagnostic, not a traded policy.


In [ ]:
def score_policy(g, name, start='1998-01', end='2025-12'):
    f, rf = g['f'].to_numpy(), g['rf'].to_numpy()
    if name == 'Buy and hold':
        net = f.copy()
    else:
        net, _ = portfolio_net_numpy(g[f'W_{name}'].to_numpy(), f, rf)
    z = pd.Series(net, index=g.index).loc[start:end]
    r_f = g['rf'].reindex(z.index)
    wealth = np.r_[1.0, np.cumprod(1 + z.to_numpy() + r_f.to_numpy())]
    if (wealth <= 0).any():
        raise ValueError('Invalid wealth path.')
    row = {'months': len(z), 'Sharpe': np.sqrt(12) * z.mean() / z.std(ddof=1),
           'objective_%_yr': 1200 * (z.mean() - GAMMA / 2 * (z ** 2).mean()),
           'vol_%_yr': 100 * np.sqrt(12) * z.std(ddof=1),
           'CAGR_%': 100 * (wealth[-1] ** (12 / len(z)) - 1),
           'max_drawdown_%': 100 * (wealth / np.maximum.accumulate(wealth) - 1).min()}
    if name != 'Buy and hold':
        x = g.loc[z.index, 'rv'] / g.loc[z.index, f'V_{name}']
        row['QLIKE_actual_ensemble'] = (x - np.log(x) - 1).mean()
    return row


rows = {}
for fac, g in results.items():
    names = ['Buy and hold', 'gjr_A', 'gjr_B', 'zhao_A_ensemble', 'zhao_B_ensemble']
    if EXTRA_ACCURACY_CONTROL:
        names.append('zhao_Aextra_ensemble')
    for name in names:
        rows[(fac, name)] = score_policy(g, name)
scorecard = pd.DataFrame(rows).T
scorecard.index.names = ['factor', 'model']
print('SMOKE OUTPUT ONLY' if SMOKE_TEST else 'New protocol: recompute inference before interpreting gains')
display(scorecard)
scorecard.to_csv(OUT / 'descriptive_scorecard.csv')


## Using the revised results in the final analysis

Use the new run folder and read the exported ensemble forecasts and weights. Do not apply the old `timing_weights()`
to those forecasts: that would recalibrate c using a different history and create a different strategy.

The following is an integration example for the final analysis notebook, **not another training cell**:

```python
g = pd.read_csv(
    'data/LSTM-Forecasts-revised/<run-id>/Mkt_forecasts.csv',
    index_col='month',
)

V_acc = g['V_zhao_A_ensemble']
V_dec = g['V_zhao_B_ensemble']
W_acc = g['W_zhao_A_ensemble']
W_dec = g['W_zhao_B_ensemble']

np.testing.assert_allclose(rf.reindex(g.index).to_numpy(), g['rf'].to_numpy())

# The existing analysis backtest is suitable only with matching 10 bp costs
# and the default zero borrowing spread. Compute before slicing the test window.
assert COST == 0.001
z_acc = backtest(W_acc, g['f'])
z_dec = backtest(W_dec, g['f'])
rv_for_accuracy = g['rv']
```

When a borrowing-spread experiment is enabled, use the same execution helper as this training notebook so that the
spread is charged consistently. Repeat for each factor file, and use the exported GJR forecasts for comparisons that
require this GJR specification. Other models in the final analysis may still need their own estimation-window alignment.

The data are revised historical files, so the annual forward test is pseudo-out-of-sample rather than a vintage-data
reconstruction. Earlier test years can enter later fitting or validation windows under the expanding historical
procedure. Chronological code does not erase choices made during prior exploration of the same sample.
